# Job Postings Analysis with LangChain + Ollama (SLMs)

**Part 2 – Role Categorization and Requirements Extraction**

For each job posting we use **LangChain** with **local small language models (SLMs) via Ollama** to extract:
1. **Job Category** – Technology/IT, Finance, Marketing, Healthcare, Education, or Other
2. **Required Skills / Technologies** – a list of skills, languages, tools
3. **Education Required** – e.g. Bachelor's, MBA, or "Not specified"
4. **Experience Required** – e.g. "3+ years", "senior-level", or "Not specified"

**Balancing speed vs accuracy (same approach as Part 1):**
- Classification is easy (short output) -> **fast tiny model** (`llama3.2:1b`).
- Requirements extraction needs more capability -> **stronger small model** (`llama3.2:3b`).
- For the full-dataset run we use **one combined call per posting** (category + skills + education + experience together) instead of several separate calls.

## Step 0: Set up Ollama (one-time)

Same setup as Part 1. Install Ollama from https://ollama.com/download, make sure it's running, then pull the models:

```bash
ollama pull llama3.2:1b
ollama pull llama3.2:3b
```

> On an Intel MacBook Pro, Ollama runs CPU-only. For a long full-dataset run, plug in, run `caffeinate -i` in a terminal so the Mac doesn't sleep, and give it airflow.

In [2]:
 !pip install langchain langchain-ollama pandas

  Using cached langchain-1.4.0-py3-none-any.whl.metadata (6.2 kB)
  Using cached langchain_ollama-1.1.0-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_core-1.6.3-py3-none-any.whl.metadata (4.8 kB)
  Using cached langgraph-1.2.11-py3-none-any.whl.metadata (4.9 kB)
  Using cached jsonpatch-1.33-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_protocol-0.0.19-py3-none-any.whl.metadata (2.4 kB)
  Using cached jsonpointer-3.1.1-py3-none-any.whl.metadata (2.4 kB)
  Using cached langgraph_checkpoint-4.2.0-py3-none-any.whl.metadata (6.7 kB)
  Using cached langgraph_prebuilt-1.1.0-py3-none-any.whl.metadata (5.2 kB)
  Using cached langgraph_sdk-0.4.4-py3-none-any.whl.metadata (5.1 kB)
  Using cached ormsgpack-1.12.2-cp314-cp314-macosx_10_12_x86_64.macosx_11_0_arm64.macosx_10_12_universal2.whl.metadata (3.2 kB)
  Using cached ollama-0.6.2-py3-none-any.whl.metadata (5.8 kB)
Using cached langchain-1.4.0-py3-none-any.whl (161 kB)
Using cached langchain_core-1.6.3-py3-none

In [3]:
import os, json, re, time
import pandas as pd

### Initialize the LLMs via LangChain + Ollama
`temperature=0` for consistent output; `num_predict` caps output length to keep generation fast.

In [4]:
from langchain_ollama import ChatOllama

FAST_MODEL    = "llama3.2:1b"   # fast; fine for a single category label
QUALITY_MODEL = "llama3.2:3b"   # stronger; better requirements extraction

classify_llm = ChatOllama(model=FAST_MODEL,    temperature=0, num_predict=12)   # tiny output -> fast
extract_llm  = ChatOllama(model=QUALITY_MODEL, temperature=0, num_predict=300)  # needs quality

# sanity check (confirms Ollama is running and the model is pulled)
print(classify_llm.invoke("Reply with just the word: ready").content)

No.


In [5]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

## Step 1: Load the Dataset

The CSV has an index column plus `Job Title` and `Job Description`. We tidy the column names, add a `Job_ID`,
and select the rows to process.

Set `NUM_JOBS`:
- `None` -> process **ALL rows**.
- a number (e.g. `25` for the base spec, or `30`) -> smaller run.

In [6]:
df_full = pd.read_csv("job_title_des.csv")
# drop the unnamed index column if present
df_full = df_full.loc[:, ~df_full.columns.str.startswith("Unnamed")]
df_full = df_full.rename(columns={"Job Title": "Job_Title", "Job Description": "Job_Description"})

print("Full dataset shape:", df_full.shape)
df_full.head()

Full dataset shape: (2277, 2)


,Job_Title,Job_Description
0,Flutter Developer,We are looking for hire experts flutter develo...
1,Django Developer,PYTHON/DJANGO (Developer/Lead) - Job Code(PDJ ...
2,Machine Learning,"Data Scientist (Contractor)\n\nBangalore, IN\n..."
3,iOS Developer,JOB DESCRIPTION:\n\nStrong framework outside o...
4,Full Stack Developer,job responsibility full stack engineer – react...


In [7]:
NUM_JOBS = 25   # None = ALL rows. Set to 25 (base spec) or 30 for a smaller run.

df = (df_full if NUM_JOBS is None else df_full.head(NUM_JOBS)).copy().reset_index(drop=True)
df["Job_ID"] = df.index + 1
df = df[["Job_ID", "Job_Title", "Job_Description"]]

print("Working dataset shape:", df.shape)
df.head()

Working dataset shape: (25, 3)


,Job_ID,Job_Title,Job_Description
0,1,Flutter Developer,We are looking for hire experts flutter develo...
1,2,Django Developer,PYTHON/DJANGO (Developer/Lead) - Job Code(PDJ ...
2,3,Machine Learning,"Data Scientist (Contractor)\n\nBangalore, IN\n..."
3,4,iOS Developer,JOB DESCRIPTION:\n\nStrong framework outside o...
4,5,Full Stack Developer,job responsibility full stack engineer – react...


Descriptions can be very long, which slows the model down. We cap the text sent to the LLM
(the full text stays in the DataFrame).

In [8]:
MAX_CHARS = 4000  # only the first ~4000 chars are sent to the model

def clip(text):
    return str(text).strip()[:MAX_CHARS]

## Step 2: Job Category Classification

A chain (`prompt | llm | parser`) that maps a posting to one broad domain, using a few-shot prompt and the
**fast** model. Falls back to `Other` if the label isn't recognized.

In [9]:
classify_system = """You are a job posting classifier.
Given a job title and description, classify the role into EXACTLY ONE domain:
Technology/IT, Finance, Marketing, Healthcare, Education, Other.

Rules:
- Reply with ONLY the domain label. No explanation.

Examples:
Title: Senior Java Developer | Backend microservices, AWS, REST APIs.
Domain: Technology/IT

Title: Financial Analyst | Budgeting, forecasting, financial modelling in Excel.
Domain: Finance

Title: Digital Marketing Manager | SEO, Google Ads, social media campaigns.
Domain: Marketing

Title: Registered Nurse | Patient care in a hospital ward, clinical duties.
Domain: Healthcare

Title: High School Mathematics Teacher | Classroom teaching and lesson planning.
Domain: Education
"""

classify_prompt = ChatPromptTemplate([
    ("system", classify_system),
    ("human", "Title: {title}\nDescription: {description}\n\nDomain:"),
])

classify_chain = classify_prompt | classify_llm | StrOutputParser()

In [10]:
CATEGORIES = ["Technology/IT", "Finance", "Marketing", "Healthcare", "Education", "Other"]

def clean_category(raw_text):
    text = raw_text.strip()
    low = text.lower()
    # check the more specific labels first
    for cat in CATEGORIES:
        if cat.lower() in low:
            return cat
    if "it" in low or "tech" in low or "software" in low:
        return "Technology/IT"
    return "Other"

def classify_job(title, description):
    raw = classify_chain.invoke({"title": title, "description": clip(description)})
    return clean_category(raw)

**Expected Output: works on a sample datapoint.**

In [11]:
sample = df.iloc[0]
print("TITLE:", sample["Job_Title"])
print("PREDICTED CATEGORY:", classify_job(sample["Job_Title"], sample["Job_Description"]))

TITLE: Flutter Developer
PREDICTED CATEGORY: Technology/IT


## Step 3: Requirements Extraction

A single composite chain that extracts **skills**, **education**, and **experience** as structured JSON,
using the **quality** model. Missing fields become `"Not specified"`. A robust parser handles imperfect JSON
from small models.

In [12]:
extract_system = """You extract structured requirements from a job description.
Return ONLY a JSON object with EXACTLY these keys:
- "skills": a JSON list of key skills, technologies, or tools (e.g. ["Python", "SQL"])
- "education": the required/preferred education level as a short string, or "Not specified"
- "experience": the required years or level of experience as a short string, or "Not specified"

Return only valid JSON, nothing else. Example:
{{"skills": ["Python", "SQL", "Power BI"], "education": "Bachelor's degree", "experience": "5+ years"}}"""

extract_prompt = ChatPromptTemplate([
    ("system", extract_system),
    ("human", "Job description:\n{description}\n\nJSON:"),
])

extract_chain = extract_prompt | extract_llm | StrOutputParser()

In [13]:
def parse_requirements(raw):
    """Parse the model output into (skills_list, education_str, experience_str)."""
    skills, education, experience = [], "Not specified", "Not specified"
    try:
        match = re.search(r"\{.*\}", raw, re.DOTALL)
        data = json.loads(match.group(0)) if match else json.loads(raw)

        s = data.get("skills", [])
        if isinstance(s, list):
            skills = [str(x).strip() for x in s if str(x).strip()]
        elif isinstance(s, str) and s.strip():
            skills = [p.strip() for p in s.split(",") if p.strip()]

        edu = str(data.get("education", "")).strip()
        education = edu if edu else "Not specified"

        exp = str(data.get("experience", "")).strip()
        experience = exp if exp else "Not specified"
    except Exception:
        pass
    return skills, education, experience

def extract_requirements(description):
    raw = extract_chain.invoke({"description": clip(description)}).strip()
    return parse_requirements(raw)

**Expected Output: works on a sample datapoint.**

In [14]:
skills, education, experience = extract_requirements(sample["Job_Description"])
print("SKILLS:    ", skills)
print("EDUCATION: ", education)
print("EXPERIENCE:", experience)

SKILLS:     ['Flutter']
EDUCATION:  Not specified
EXPERIENCE: 1 year


## Step 4: Apply the LLM Chain to Each Job Posting (efficient combined call)

To make a full run practical, we combine classification + all three extractions into **one call per posting**
that returns a single JSON object (`category`, `skills`, `education`, `experience`). If the combined JSON
fails to parse for a posting, we fall back to the separate chains for that one row.

In [16]:
analyze_llm = ChatOllama(model=QUALITY_MODEL, temperature=0, num_predict=380)

analyze_system = """You analyze a job posting and return ONLY a JSON object with EXACTLY these keys:
- "category": exactly one of Technology/IT, Finance, Marketing, Healthcare, Education, Other
- "skills": a JSON list of key skills, technologies, or tools
- "education": required/preferred education level as a short string, or "Not specified"
- "experience": required years or level of experience as a short string, or "Not specified"

Return only valid JSON and nothing else. Example:
{{"category": "Technology/IT", "skills": ["Python", "SQL"], "education": "Bachelor's degree", "experience": "3+ years"}}"""

analyze_prompt = ChatPromptTemplate([
    ("system", analyze_system),
    ("human", "Job title: {title}\nJob description:\n{description}\n\nJSON:"),
])

analyze_chain = analyze_prompt | analyze_llm | StrOutputParser()

def analyze_job(title, description):
    """Return (category, skills, education, experience) from a single combined call."""
    raw = analyze_chain.invoke({"title": title, "description": clip(description)}).strip()
    try:
        match = re.search(r"\{.*\}", raw, re.DOTALL)
        data = json.loads(match.group(0)) if match else json.loads(raw)
        category = clean_category(str(data.get("category", "Other")))
        skills, education, experience = parse_requirements(raw)
        return category, skills, education, experience
    except Exception:
        # fall back to the separate chains for this posting
        category = classify_job(title, description)
        skills, education, experience = extract_requirements(description)
        return category, skills, education, experience

### Run over every posting
Progress prints every 10 rows with elapsed time and ETA; results are checkpointed to CSV every 100 rows.

In [17]:
OUTPUT_CSV       = "job_analysis_full.csv"
PRINT_EVERY      = 10
CHECKPOINT_EVERY = 100

categories, skills_list, educations, experiences = [], [], [], []
total = len(df)
start = time.time()

for i, row in df.iterrows():
    try:
        cat, sk, edu, exp = analyze_job(row["Job_Title"], row["Job_Description"])
    except Exception as e:
        print(f"Row {i+1} error: {e}")
        cat, sk, edu, exp = "Other", [], "Not specified", "Not specified"

    categories.append(cat)
    skills_list.append(sk)
    educations.append(edu)
    experiences.append(exp)

    done = i + 1
    if done % PRINT_EVERY == 0 or done == total:
        elapsed = time.time() - start
        rate = done / elapsed
        eta = (total - done) / rate if rate > 0 else 0
        print(f"[{done}/{total}] {rate:.2f} job/s | elapsed {elapsed/60:.1f} min | ETA {eta/60:.1f} min")

    if done % CHECKPOINT_EVERY == 0:
        ckpt = df.iloc[:done].copy()
        ckpt["Predicted_Category"]   = categories
        ckpt["Required_Skills"]      = [json.dumps(s) for s in skills_list]
        ckpt["Education_Required"]   = educations
        ckpt["Experience_Required"]  = experiences
        ckpt.to_csv(OUTPUT_CSV, index=False)

print("\nDone. Total time:", round((time.time() - start) / 60, 1), "minutes")

[10/25] 0.15 job/s | elapsed 1.1 min | ETA 1.7 min
[20/25] 0.15 job/s | elapsed 2.3 min | ETA 0.6 min
[25/25] 0.15 job/s | elapsed 2.8 min | ETA 0.0 min

Done. Total time: 2.8 minutes


## Step 5: Update the DataFrame with New Columns

In [18]:
df["Predicted_Category"]  = categories
df["Required_Skills"]     = skills_list           # kept as Python lists
df["Education_Required"]   = educations
df["Experience_Required"]  = experiences

# final save (skills stored as JSON text so lists survive a CSV round-trip)
save_df = df.copy()
save_df["Required_Skills"] = save_df["Required_Skills"].apply(json.dumps)
save_df.to_csv(OUTPUT_CSV, index=False)
print("Saved", len(df), "rows to", OUTPUT_CSV)

Saved 25 rows to job_analysis_full.csv


### The new columns only

In [19]:
df[["Job_ID", "Predicted_Category", "Required_Skills", "Education_Required", "Experience_Required"]].head(10)

,Job_ID,Predicted_Category,Required_Skills,Education_Required,Experience_Required
0,1,Technology/IT,"[Flutter, Software Development]",Not specified,1 year
1,2,Technology/IT,"[Python, Django, API development, SQL, PyUnit,...",Not specified,Not specified
2,3,Technology/IT,"[Python, Java, Machine Learning, Deep Learning...",M.Sc. in Computer Science,3+ years
3,4,Technology/IT,"[Objective-C, Cocoa Touch, Core Data, Core Ani...",Not specified,Not specified
4,5,Technology/IT,"[React, JavaScript, HTML, CSS, Redux, Angular,...",Not specified,5+ years
5,6,Technology/IT,"[C#, .NET, .NET Core, HTML5, CSS3, MsSQL, MySQ...","Bachelor's Degree in Computer Science, Informa...",2 years
6,7,Technology/IT,"[NodeJS, Java, MongoDB, Elasticsearch, Redis, ...",B.Sc,2+ years
7,8,Technology/IT,"[ReactJS, NodeJS, Azure Functions, GraphQL, HT...",Not specified,3 - 8 years
8,9,Technology/IT,"[Bash, Ruby, Python, Java, Puppet, Chef, Cloud...",Not specified,Not specified
9,10,Technology/IT,"[REST API, C/C++, Python, Go, Git, Gerrit, Jen...","BS or MS; computer engineering, computer scien...",7+ years


### Final DataFrame: original + new columns together

In [ ]:
df.head(25)

### Category distribution (quick spot-check)

In [20]:
df["Predicted_Category"].value_counts()

Predicted_Category
Technology/IT    24
Finance           1
Name: count, dtype: int64

## Output as JSON

Each record follows the requested structure (`Job_Title`, `Job_Description`, `Predicted_Category`,
`Required_Skills`, `Education_Required`, `Experience_Required`), with a short description excerpt to keep
the file readable.

In [21]:
json_df = df.copy()
json_df["Job_Description"] = json_df["Job_Description"].str.strip().str.slice(0, 200) + "..."

records = json_df[
    ["Job_Title", "Job_Description", "Predicted_Category",
     "Required_Skills", "Education_Required", "Experience_Required"]
].to_dict(orient="records")

# show the first record like the assignment example
print(json.dumps(records[0], indent=2, ensure_ascii=False))

{
  "Job_Title": "Flutter Developer",
  "Job_Description": "We are looking for hire experts flutter developer. So you are eligible this post then apply your resume.\nJob Types: Full-time, Part-time\nSalary: ₹20,000.00 - ₹40,000.00 per month\nBenefits:\nFlexible sc...",
  "Predicted_Category": "Technology/IT",
  "Required_Skills": [
    "Flutter",
    "Software Development"
  ],
  "Education_Required": "Not specified",
  "Experience_Required": "1 year"
}


In [22]:
with open("job_analysis_output.json", "w", encoding="utf-8") as f:
    json.dump(records, f, indent=2, ensure_ascii=False)

print(f"Saved {len(records)} records to job_analysis_output.json")

Saved 25 records to job_analysis_output.json
